In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("demo").getOrCreate()


In [3]:
file_name = 'transaction_detail.csv'

In [5]:
df = spark.read.csv(file_name,header=True,inferSchema=True)
df.show(10)

+------------------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
|transaction_amount|transaction_card_type|transaction_ecommerce_website_name|transaction_country_name|transaction_datetime|transaction_id|transaction_city_name|transaction_product_name|
+------------------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
|             50.85|           MasterCard|                      www.ebay.com|                   India| 2019-05-14 15:24:12|             1|               Mumbai|                  Laptop|
|            259.12|           MasterCard|                    www.amazon.com|                   India| 2019-05-14 15:24:13|             2|                 Pune|              Wrist Band|
|            328.16|           MasterCard|                  www.flipka

In [6]:
df.printSchema()

root
 |-- transaction_amount: double (nullable = true)
 |-- transaction_card_type: string (nullable = true)
 |-- transaction_ecommerce_website_name: string (nullable = true)
 |-- transaction_country_name: string (nullable = true)
 |-- transaction_datetime: timestamp (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- transaction_city_name: string (nullable = true)
 |-- transaction_product_name: string (nullable = true)



# Select

In [7]:
df.select('transaction_product_name','transaction_amount').show(10)

+------------------------+------------------+
|transaction_product_name|transaction_amount|
+------------------------+------------------+
|                  Laptop|             50.85|
|              Wrist Band|            259.12|
|                TV Stand|            328.16|
|                TV Stand|            399.06|
|     External Hard Drive|            194.52|
|                      TV|            415.65|
|             Wrist Watch|            467.12|
|                  Laptop|            361.61|
|              Text Books|            357.34|
|                  Laptop|            495.91|
+------------------------+------------------+
only showing top 10 rows


# Rename or add Columns

In [8]:
df = df.withColumnRenamed('transaction_amount','amount')
df.show(10)

+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
|amount|transaction_card_type|transaction_ecommerce_website_name|transaction_country_name|transaction_datetime|transaction_id|transaction_city_name|transaction_product_name|
+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
| 50.85|           MasterCard|                      www.ebay.com|                   India| 2019-05-14 15:24:12|             1|               Mumbai|                  Laptop|
|259.12|           MasterCard|                    www.amazon.com|                   India| 2019-05-14 15:24:13|             2|                 Pune|              Wrist Band|
|328.16|           MasterCard|                  www.flipkart.com|           United States| 2019-05-14 15:24:14|             3|    

In [9]:
from pyspark.sql.functions import upper

df = df.withColumn('product_name', upper(df.transaction_product_name)).limit(10)
df.show(10)

+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+-------------------+
|amount|transaction_card_type|transaction_ecommerce_website_name|transaction_country_name|transaction_datetime|transaction_id|transaction_city_name|transaction_product_name|       product_name|
+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+-------------------+
| 50.85|           MasterCard|                      www.ebay.com|                   India| 2019-05-14 15:24:12|             1|               Mumbai|                  Laptop|             LAPTOP|
|259.12|           MasterCard|                    www.amazon.com|                   India| 2019-05-14 15:24:13|             2|                 Pune|              Wrist Band|         WRIST BAND|
|328.16|           MasterCard|

In [10]:
df = df.drop('product_name')
df.show(10)

+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
|amount|transaction_card_type|transaction_ecommerce_website_name|transaction_country_name|transaction_datetime|transaction_id|transaction_city_name|transaction_product_name|
+------+---------------------+----------------------------------+------------------------+--------------------+--------------+---------------------+------------------------+
| 50.85|           MasterCard|                      www.ebay.com|                   India| 2019-05-14 15:24:12|             1|               Mumbai|                  Laptop|
|259.12|           MasterCard|                    www.amazon.com|                   India| 2019-05-14 15:24:13|             2|                 Pune|              Wrist Band|
|328.16|           MasterCard|                  www.flipkart.com|           United States| 2019-05-14 15:24:14|             3|    

# select distinct values

In [11]:
df.select('transaction_card_type','transaction_country_name').distinct().show(10)

+---------------------+------------------------+
|transaction_card_type|transaction_country_name|
+---------------------+------------------------+
|           MasterCard|                   India|
|           MasterCard|           United States|
|                 Visa|                   Inida|
|                 Visa|                   Italy|
|              Maestro|                   India|
|                 Visa|                   India|
+---------------------+------------------------+



# Group By and Order by

In [12]:
from pyspark.sql.functions import col, sum, count, mean
grouped_data = df.groupBy(["transaction_card_type", "transaction_country_name"]).agg(
    sum("amount").alias("total amount"),
    count("transaction_card_type").alias("total transection"),
    mean("amount").alias("avg amount")
)

grouped_data.show()

+---------------------+------------------------+------------+-----------------+------------------+
|transaction_card_type|transaction_country_name|total amount|total transection|        avg amount|
+---------------------+------------------------+------------+-----------------+------------------+
|           MasterCard|                   India|      777.09|                3|259.03000000000003|
|           MasterCard|           United States|      328.16|                1|            328.16|
|                 Visa|                   Inida|      399.06|                1|            399.06|
|                 Visa|                   Italy|      194.52|                1|            194.52|
|              Maestro|                   India|      772.99|                2|           386.495|
|                 Visa|                   India|      857.52|                2|            428.76|
+---------------------+------------------------+------------+-----------------+------------------+



In [13]:
ordered_data = grouped_data.orderBy("transaction_country_name", "transaction_card_type")
ordered_data.show()

+---------------------+------------------------+------------+-----------------+------------------+
|transaction_card_type|transaction_country_name|total amount|total transection|        avg amount|
+---------------------+------------------------+------------+-----------------+------------------+
|              Maestro|                   India|      772.99|                2|           386.495|
|           MasterCard|                   India|      777.09|                3|259.03000000000003|
|                 Visa|                   India|      857.52|                2|            428.76|
|                 Visa|                   Inida|      399.06|                1|            399.06|
|                 Visa|                   Italy|      194.52|                1|            194.52|
|           MasterCard|           United States|      328.16|                1|            328.16|
+---------------------+------------------------+------------+-----------------+------------------+



In [15]:
ordered_data = grouped_data.orderBy(["transaction_country_name", "transaction_card_type"],ascending=[False, True])
ordered_data.show()

+---------------------+------------------------+------------+-----------------+------------------+
|transaction_card_type|transaction_country_name|total amount|total transection|        avg amount|
+---------------------+------------------------+------------+-----------------+------------------+
|           MasterCard|           United States|      328.16|                1|            328.16|
|                 Visa|                   Italy|      194.52|                1|            194.52|
|                 Visa|                   Inida|      399.06|                1|            399.06|
|              Maestro|                   India|      772.99|                2|           386.495|
|           MasterCard|                   India|      777.09|                3|259.03000000000003|
|                 Visa|                   India|      857.52|                2|            428.76|
+---------------------+------------------------+------------+-----------------+------------------+

